# RoBERTa-CRF-BoundaryAttention vs. **LUKE-base (entity-aware span classification)** — Fair Fold-2 Comparison

This notebook is derived from your `roberta_crf_vs_modernbert_fold2_comparison.ipynb`. It keeps
**every** piece of that pipeline identical (dataset loading, sliding-window fold construction, BIO
label scheme for the RoBERTa-CRF model, training hyperparameters, seqeval/nervaluate metrics, and the
RoBERTa-CRF-BoundaryAttention checkpoint evaluation) and swaps out the LUKE side to use LUKE's own
**entity-aware span classification** head, `LukeForEntitySpanClassification`, instead of plain
token classification.

## Two things fixed / changed while porting your notebook

1. **Hard-coded HF token removed.** Your original notebook had a live `HF_TOKEN` pasted in plaintext
   in a cell. If you haven't already, revoke it at https://huggingface.co/settings/tokens and use
   Colab's Secrets tab or the interactive `getpass` prompt below instead. This notebook never
   hard-codes a token.
2. **`FOLD_IDX` set to 2, matching your notebook's code cell.** Your uploaded notebook's markdown
   described "Fold 2 (1-indexed)" as `fold_idx = 1`, while the code cell itself used `folds[2]`. Per
   your explicit instruction, this notebook uses **`FOLD_IDX = 2`** — i.e. it now matches the code, not
   the earlier markdown description. If `folds[2]` is *not* actually what produced the
   `Souvikbasur/roberta-crf-news-ner-kfold-best` checkpoint in your k-fold training run, this
   comparison is no longer apples-to-apples — double check against that notebook before trusting the
   numbers below. Everything downstream is parameterized off the one `FOLD_IDX` constant.

## What changed: `LukeForEntitySpanClassification` instead of `LukeForTokenClassification`

Per your request, LUKE is now run in its **original entity-aware span classification** mode — the mode
LUKE's own paper and official fine-tuning scripts use for NER — instead of the generic
`AutoModelForTokenClassification`-style head used in the previous version of this notebook. Concretely:

- **Candidate spans, not per-token tags.** The sentence is split into words (`split_words_with_offsets`),
  and every contiguous span of 1..`MAX_MENTION_LENGTH` words becomes a *candidate*. LUKE's entity-aware
  encoder embeds each candidate span directly (via its entity vocabulary / entity position embeddings),
  and `LukeForEntitySpanClassification`'s classification head predicts one label per candidate: one of
  the 17 entity types, or `O` ("not an entity"). This is a fundamentally different labeling scheme from
  BIO tagging — there's no `B-`/`I-` distinction because a span already carries its own boundaries.
- **Greedy non-maximum suppression at decode time.** Because candidate spans overlap by construction
  (e.g. "New York" and "New York City" are both candidates), the model's raw per-span predictions can
  disagree about overlapping regions. `decode_entity_spans` (section 9) resolves this the standard way:
  keep the highest-confidence non-`O` span, discard every remaining candidate that overlaps it, repeat.
  This produces a final set of non-overlapping predicted spans, directly comparable to the gold entities.
- **BIO conversion happens only for scoring, after decoding — not during modeling.** Your existing
  seqeval / nervaluate pipeline expects BIO tag sequences. Since span classification's native output is
  spans, not tags, `bio_tags_from_spans` (section 4) converts BOTH the gold entities and LUKE's final
  decoded spans into a word-level BIO sequence purely for scoring purposes, so the same seqeval/nervaluate
  calls your notebook already used for the CRF model can be reused unchanged for LUKE. The
  error-taxonomy pipeline (`run_error_analysis`, section 4) works directly off the raw decoded spans —
  it never needed BIO tags in the first place.
- **Negative sampling only touches TRAIN/VAL, never TEST.** Enumerating all word-spans up to length 16
  produces far more `O` candidates than positive entities per sentence. To keep training tractable,
  `TRAIN_NEG_SAMPLE_RATIO` randomly downsamples negative candidates *for training and the early-stopping
  validation set only* — every positive (gold-entity) span is always kept. The held-out **Fold 2 test
  set is scored on every single candidate span, with no sampling and no truncation** (chunked into
  batches of `MAX_ENTITY_LENGTH` at inference time instead) — so this cannot inflate the reported
  numbers; it only makes training feasible.

## What's kept byte-for-byte identical to your notebook

- `parse_label_studio`, `build_folds`, `train_val_split_within_pool` — the sliding-window Fold 2
  reproduction logic
- `tokenize_and_align`, `NERDataset` — used for the RoBERTa-CRF model, unchanged fast-tokenizer path
- `SEED=42`, `K_FOLDS=5`, `VAL_FRACTION_WITHIN_TRAIN=0.10`, `MAX_LENGTH=256`
- `LEARNING_RATE`, `BATCH_SIZE`, `GRAD_ACCUM_STEPS`, `NUM_EPOCHS`, `WARMUP_RATIO`, `WEIGHT_DECAY`,
  `EARLY_STOP_PAT`
- `BoundaryGuidedAttention`, `RobertaCRFBoundaryAttentionForNER`, `BoundaryAttentionTrainer` — needed to
  load and evaluate your existing checkpoint, unchanged
- seqeval P/R/F1/accuracy, `nervaluate` strict F1, and the full `run_error_analysis` confusion /
  boundary-error taxonomy — reused unchanged, fed by a new `predict_fn` wrapper for LUKE, exactly the
  same pattern your notebook used for the CRF vs. ModernBERT split

## What's necessarily different for LUKE

- `LukeForEntitySpanClassification` instead of a plain token-classification head, trained on
  candidate-span labels (`span_label2id`), not BIO labels (`label2id` is now RoBERTa-CRF-only)
- Word-level candidate-span enumeration (`split_words_with_offsets`, `enumerate_candidate_spans`)
  instead of sub-word BIO alignment — LUKE's span classifier reasons over words, not sub-tokens
- Greedy NMS decoding (`decode_entity_spans`) to resolve overlapping span predictions into a final
  non-overlapping set, then `bio_tags_from_spans` to convert those (and the gold entities) into
  word-level BIO tags purely for seqeval/nervaluate scoring


In [1]:
# One clean install. transformers>=4.51 is required for the current LukeForEntitySpanClassification
# checkpoint-compatible implementation; everything else matches your original notebook's dependency set.
!pip install -q -U "transformers>=4.51.0" torch pytorch-crf seqeval nervaluate huggingface_hub safetensors pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 113.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 10.0 MB/s eta 0:00:00
  

In [3]:
!pip install -q \
transformers>=4.51.0 \
pytorch-crf \
seqeval \
nervaluate \
huggingface_hub \
safetensors

In [1]:
import torch
import torchvision
import transformers

print(torch.__version__)
print(torchvision.__version__)
print(transformers.__version__)

2.11.0+cu128
0.26.0+cu128
4.57.1


In [2]:
!pip uninstall -y transformers
!pip install -U transformers==4.57.1

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
!pip install -q pytorch-crf seqeval nervaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [4]:
import json, random, os, getpass
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModel, LukeTokenizer, LukeForEntitySpanClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
from torchcrf import CRF
from seqeval.metrics import (
    classification_report, f1_score, precision_score, recall_score, accuracy_score,
)
from huggingface_hub import hf_hub_download

# ---- unchanged from your notebook ----
BASE_MODEL       = "FacebookAI/roberta-base"          # RoBERTa-CRF-BoundaryAttention backbone
DATA_PATH        = "17entity_news_ner_dataset.json"   # <-- upload the SAME file your original notebook used
OUTPUT_DIR       = "./ner_fold2_luke_comparison_output"
MAX_LENGTH       = 256
SEED             = 42
K_FOLDS                   = 5
VAL_FRACTION_WITHIN_TRAIN = 0.10

LEARNING_RATE    = 1e-5
BATCH_SIZE       = 8
GRAD_ACCUM_STEPS = 4
NUM_EPOCHS       = 3
WARMUP_RATIO     = 0.06
WEIGHT_DECAY     = 0.01
EARLY_STOP_PAT   = 2

BOUNDARY_LOSS_WEIGHT = 0.3   # only used by the RoBERTa-CRF-BoundaryAttention class (for loading it)
ATTN_HEADS           = 4

device = "cuda" if torch.cuda.is_available() else "cpu"

ENTITY_TAGS = [
    "PER", "ORG", "LOC", "FAC", "DATE", "TIME", "EVT",
    "PROD", "MONEY", "PCT", "LAW", "NORP", "LANG",
    "WOA", "QTY", "ORD", "CARD",
]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

def build_label_maps():
    labels = ["O"]
    for tag in ENTITY_TAGS:
        labels.append(f"B-{tag}")
        labels.append(f"I-{tag}")
    label2id = {l: i for i, l in enumerate(labels)}
    id2label = {i: l for l, i in label2id.items()}
    return label2id, id2label

label2id, id2label = build_label_maps()

# ---- new for this comparison ----
FOLD_NUMBER = 2                   # 1-indexed, matches the original notebook's printed "FOLD 2 / 5"
FOLD_IDX    = 2                   # 0-indexed into `folds` -- CHANGED per explicit instruction: set to
                                   # match the code cell in your uploaded notebook (`folds[2]`), not the
                                   # markdown's "fold_idx = 1" description. If `folds[2]` is NOT what
                                   # actually produced the `roberta-crf-news-ner-kfold-best` checkpoint,
                                   # this comparison silently stops being apples-to-apples -- re-verify
                                   # against your k-fold training notebook before trusting the numbers.

LUKE_MODEL_NAME = "studio-ousia/luke-base"                             # baseline backbone, no CRF/extra heads
CRF_MODEL_REPO  = "Souvikbasur/roberta-crf-news-ner-kfold-best"        # your uploaded fold-2 checkpoint

# ---- new: LukeForEntitySpanClassification config ----
# LUKE's entity-aware span classifier reasons over WORD-level candidate spans (start word, end
# word), not BIO tags per sub-word token. Every contiguous span of 1..MAX_MENTION_LENGTH words is
# a "candidate"; the model classifies each candidate as one of ENTITY_TAGS or "O" (not-an-entity).
MAX_MENTION_LENGTH     = 16    # max candidate span length in WORDS. Gold entities longer than this
                                # can never be predicted correctly by construction -- checked explicitly
                                # in the verification cell (section 6) before any training happens.
MAX_ENTITY_LENGTH      = 128   # max candidate spans processed in a single forward pass (LUKE's entity
                                # side has its own position-embedding table; long sentences can produce
                                # far more than 128 candidates, so eval chunks them -- see section 9).
TRAIN_NEG_SAMPLE_RATIO = 4     # TRAIN/VAL ONLY: for every positive (true-entity) candidate span in a
                                # sentence, keep at most this many randomly-sampled negative ("O")
                                # candidates, to keep the O:entity class imbalance and forward-pass count
                                # manageable. Every positive span is always kept (never sampled away).
                                # The held-out TEST set is NEVER negative-sampled or truncated -- it
                                # always scores every single candidate span, so this has no way to
                                # inflate the reported test metrics.

SPAN_ENTITY_TAGS = ["O"] + ENTITY_TAGS                       # span-classification label set: no B-/I-
                                                               # distinction needed since a span already
                                                               # carries its own boundaries.
span_label2id = {l: i for i, l in enumerate(SPAN_ENTITY_TAGS)}
span_id2label = {i: l for l, i in span_label2id.items()}

print(f"Device: {device} | Total label classes: {len(label2id)} | K_FOLDS: {K_FOLDS} | "
      f"Reproducing Fold {FOLD_NUMBER} (fold_idx={FOLD_IDX})")
print(f"Span-classification labels: {len(span_label2id)} ({SPAN_ENTITY_TAGS})")


Device: cuda | Total label classes: 35 | K_FOLDS: 5 | Reproducing Fold 2 (fold_idx=2)
Span-classification labels: 18 (['O', 'PER', 'ORG', 'LOC', 'FAC', 'DATE', 'TIME', 'EVT', 'PROD', 'MONEY', 'PCT', 'LAW', 'NORP', 'LANG', 'WOA', 'QTY', 'ORD', 'CARD'])


## 1. Load the dataset and build the same sliding-window folds

`parse_label_studio`, `build_folds`, `train_val_split_within_pool`, and `tokenize_and_align` below are
byte-for-byte copies of the functions in your comparison notebook — same BIO-labeling fix (`B-` only
fires on the token whose `char_start` matches the entity's own start offset), same fold-construction
logic. `tokenize_and_align` is the **fast-tokenizer** path and is used only for the RoBERTa-CRF model
below (section 8); LUKE gets its own tokenization function in section 5 for the reasons explained above.


In [13]:
def parse_label_studio(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    tasks, skipped, seen_texts = [], 0, set()
    for item in raw:
        text = item.get("data", {}).get("text", "").strip()
        if not text or text in seen_texts:
            skipped += 1
            continue
        seen_texts.add(text)
        entities = []
        annotations = item.get("annotations", [])
        if annotations:
            for result in annotations[0].get("result", []):
                val = result.get("value", {})
                start, end, labels = val.get("start"), val.get("end"), val.get("labels", [])
                if start is None or end is None or not labels:
                    continue
                if labels[0] in ENTITY_TAGS:
                    entities.append((start, end, labels[0]))
        tasks.append({"text": text, "entities": entities})
    print(f"[Parse] Loaded {len(tasks)} tasks, skipped {skipped}.")
    return tasks

# --- NEW: defined here (not in the LUKE section) because it's plain offset cleanup with no
# dependency on split_words_with_offsets / candidate spans, and it must run before build_folds
# so the fix is baked into every fold, for both models. ---
def normalize_entity_offsets(tasks):
    """Strips leading/trailing whitespace from every gold entity span's character offsets.
    Applied identically to train/val/test -- looks only at the raw text inside the existing
    span, never at labels or model output, so it can't leak information, only fix bookkeeping."""
    n_adjusted = 0
    fixed_tasks = []
    for task in tasks:
        text = task["text"]
        new_entities = []
        for (start, end, label) in task["entities"]:
            new_start, new_end = start, end
            while new_end > new_start and text[new_end - 1].isspace():
                new_end -= 1
            while new_start < new_end and text[new_start].isspace():
                new_start += 1
            if (new_start, new_end) != (start, end):
                n_adjusted += 1
            if new_end > new_start:
                new_entities.append((new_start, new_end, label))
        fixed_tasks.append({"text": text, "entities": new_entities})
    print(f"[Offset cleanup] {n_adjusted} gold entity span(s) had leading/trailing whitespace "
          f"trimmed from their character offsets.")
    return fixed_tasks

def build_folds(tasks, k=K_FOLDS, seed=SEED):
    """Sliding-window k-fold split: shuffle once (seeded, reproducible), then
    cut the shuffled list into k contiguous, roughly-equal windows. Each
    window takes a turn as the held-out TEST fold while the rest become the
    train+val pool for that fold -- so across all k folds, every sentence is
    test-evaluated exactly once, by a model that never saw it in training."""
    shuffled = tasks[:]
    random.Random(seed).shuffle(shuffled)
    n = len(shuffled)
    fold_size = n // k
    folds, start = [], 0
    for i in range(k):
        end = start + fold_size if i < k - 1 else n   # last fold absorbs the remainder
        folds.append(shuffled[start:end])
        start = end
    sizes = [len(f) for f in folds]
    print(f"[Fold build] {k} folds from {n} sentences, sizes: {sizes}")
    assert sum(sizes) == n, "fold sizes must sum to the full dataset"
    return folds


def train_val_split_within_pool(pool, val_fraction=VAL_FRACTION_WITHIN_TRAIN, seed=SEED):
    p = pool[:]
    random.Random(seed).shuffle(p)
    n_val = int(len(p) * val_fraction)
    return p[n_val:], p[:n_val]    # train_tasks, val_tasks


def tokenize_and_align(task_list, tokenizer, label2id, max_length=MAX_LENGTH):
    """Unchanged from your notebook -- fast-tokenizer path (used for RoBERTa-CRF only here).
    B- only fires on the token whose char_start matches the entity's own start offset. Every other
    token of the same entity gets I-, even if it starts a new tokenizer word."""
    encoded = []
    for task in task_list:
        text = task["text"]
        entities = sorted(task["entities"], key=lambda x: x[0])
        tok = tokenizer(text, max_length=max_length, truncation=True,
                         padding="max_length", return_offsets_mapping=True, return_tensors=None)
        offsets = tok["offset_mapping"]
        word_ids = tok.word_ids()
        labels, b_start, b_end, prev_word_id = [], [], [], None

        for offset, wid in zip(offsets, word_ids):
            if wid is None:
                labels.append(-100); b_start.append(-100); b_end.append(-100)
                prev_word_id = wid
                continue
            char_start, char_end = offset
            matched_label, matched_entity_start, matched_entity_end = "O", None, None
            for (ent_start, ent_end, ent_label) in entities:
                if char_start >= ent_start and char_end <= ent_end and char_end > char_start:
                    matched_label, matched_entity_start, matched_entity_end = ent_label, ent_start, ent_end
                    break

            if matched_label == "O":
                bio = "O"
            elif wid != prev_word_id and char_start == matched_entity_start:
                bio = f"B-{matched_label}"
            else:
                bio = f"I-{matched_label}"
            labels.append(label2id.get(bio, label2id["O"]))

            is_start = 1 if (matched_label != "O" and char_start == matched_entity_start) else 0
            is_end = 1 if (matched_label != "O" and char_end == matched_entity_end) else 0
            b_start.append(is_start)
            b_end.append(is_end)
            prev_word_id = wid

        encoded.append({
            "input_ids": tok["input_ids"], "attention_mask": tok["attention_mask"],
            "labels": labels, "boundary_start": b_start, "boundary_end": b_end,
        })
    return encoded


class NERDataset(Dataset):
    """Full dataset (with boundary_start/boundary_end) -- used for the RoBERTa-CRF-BoundaryAttention
    model, which consumes those fields."""
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        e = self.encodings[idx]
        return {
            "input_ids": torch.tensor(e["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(e["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(e["labels"], dtype=torch.long),
            "boundary_start": torch.tensor(e["boundary_start"], dtype=torch.long),
            "boundary_end": torch.tensor(e["boundary_end"], dtype=torch.long),
        }


set_seed(SEED)
tasks = parse_label_studio(DATA_PATH)
tasks = normalize_entity_offsets(tasks)          # <-- new line, only addition to this cell
folds = build_folds(tasks, k=K_FOLDS, seed=SEED)

[Parse] Loaded 24044 tasks, skipped 822.
[Offset cleanup] 960 gold entity span(s) had leading/trailing whitespace trimmed from their character offsets.
[Fold build] 5 folds from 24044 sentences, sizes: [4808, 4808, 4808, 4808, 4812]


## 2. Reproduce the exact Fold 2 split (no new split, no reshuffle)

This mirrors the body of the original k-fold loop for `fold_idx = FOLD_IDX` only — it does not run the
other 4 folds, and it does not touch `random`'s state differently than the original loop would have at
this point (each call below reseeds with the fixed `SEED`, exactly like `train_val_split_within_pool`
does internally).


In [14]:
test_tasks = folds[FOLD_IDX]
train_pool = [t for i, f in enumerate(folds) if i != FOLD_IDX for t in f]
train_tasks, val_tasks = train_val_split_within_pool(train_pool)

print(f"Fold {FOLD_NUMBER} reproduced (fold_idx={FOLD_IDX}):")
print(f"  Train: {len(train_tasks)} | Val (early-stopping monitor): {len(val_tasks)} | "
      f"Test (held-out window): {len(test_tasks)}")

# --- sanity checks: disjointness + full coverage of the (K-1)/K pool ---
train_texts = {t["text"] for t in train_tasks}
val_texts   = {t["text"] for t in val_tasks}
test_texts  = {t["text"] for t in test_tasks}
assert train_texts.isdisjoint(val_texts), "train/val overlap -- split is not reproducing correctly"
assert train_texts.isdisjoint(test_texts), "train/test overlap -- split is not reproducing correctly"
assert val_texts.isdisjoint(test_texts), "val/test overlap -- split is not reproducing correctly"
assert len(train_texts) + len(val_texts) == len(train_pool), "train+val must cover the whole pool"
print("Sanity check passed: train/val/test are disjoint and train+val fully covers the non-test pool.")


Fold 2 reproduced (fold_idx=2):
  Train: 17313 | Val (early-stopping monitor): 1923 | Test (held-out window): 4808
Sanity check passed: train/val/test are disjoint and train+val fully covers the non-test pool.


## 3. RoBERTa + CRF + Boundary Attention — model class (needed to load your checkpoint)

Unchanged from your notebook, verbatim (`BoundaryGuidedAttention`, `RobertaCRFBoundaryAttentionForNER`).
We need this class defined in this session purely to load the state dict you uploaded to
`Souvikbasur/roberta-crf-news-ner-kfold-best` — **we do not train this model**, only evaluate the
checkpoint you already trained.


In [15]:
class BoundaryGuidedAttention(nn.Module):
    """Multi-head self-attention where the attention logits get an additive
    bias from each KEY token's boundary evidence, so tokens carrying strong
    boundary signal pull more attention from every other token in the
    sequence -- not just their immediate neighbors. Padding keys are masked
    out entirely. Unchanged from the single-split notebook."""

    def __init__(self, hidden_size, num_heads=4):
        super().__init__()
        assert hidden_size % num_heads == 0, "hidden_size must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        self.out_proj = nn.Linear(hidden_size, hidden_size)
        self.bias_scale = nn.Parameter(torch.tensor(1.0))
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(0.1)

    def forward(self, H, boundary_score, attention_mask):
        batch, seq_len, hidden = H.shape

        def split_heads(x):
            return x.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        Q = split_heads(self.q_proj(H))
        K = split_heads(self.k_proj(H))
        V = split_heads(self.v_proj(H))

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)

        key_bias = (self.bias_scale * boundary_score).unsqueeze(1).unsqueeze(1)
        scores = scores + key_bias

        key_mask = attention_mask.bool().unsqueeze(1).unsqueeze(1)
        scores = scores.masked_fill(~key_mask, float("-inf"))

        attn = torch.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn)
        attn = self.dropout(attn)

        context = torch.matmul(attn, V)
        context = context.transpose(1, 2).contiguous().view(batch, seq_len, hidden)
        context = self.out_proj(context)

        return self.layer_norm(H + context)


class RobertaCRFBoundaryAttentionForNER(nn.Module):
    def __init__(self, base_model_name, num_labels, boundary_loss_weight=BOUNDARY_LOSS_WEIGHT,
                 attn_heads=ATTN_HEADS):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(base_model_name)
        hidden = self.roberta.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden, num_labels)
        self.crf = CRF(num_labels, batch_first=True)
        self.num_labels = num_labels

        self.boundary_start_head = nn.Linear(hidden, 1)
        self.boundary_end_head = nn.Linear(hidden, 1)
        self.boundary_attention = BoundaryGuidedAttention(hidden, num_heads=attn_heads)

        self.boundary_loss_weight = boundary_loss_weight

    def forward(self, input_ids, attention_mask, labels=None,
                boundary_start=None, boundary_end=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        H = outputs.last_hidden_state

        start_logits = self.boundary_start_head(H).squeeze(-1)
        end_logits = self.boundary_end_head(H).squeeze(-1)
        boundary_score = torch.sigmoid(start_logits + end_logits)

        H_context = self.boundary_attention(H, boundary_score, attention_mask)

        emissions = self.classifier(self.dropout(H_context))
        mask = attention_mask.bool()

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0
            crf_mask = mask & (labels != -100)
            crf_mask[:, 0] = True
            log_likelihood = self.crf(emissions, safe_labels, mask=crf_mask, reduction="mean")
            crf_loss = -log_likelihood

            valid = (boundary_start != -100).float()
            bs_target = (boundary_start == 1).float() * attention_mask.float()
            be_target = (boundary_end == 1).float() * attention_mask.float()
            bs_loss = F.binary_cross_entropy_with_logits(start_logits, bs_target, reduction="none")
            be_loss = F.binary_cross_entropy_with_logits(end_logits, be_target, reduction="none")
            bs_loss = (bs_loss * valid).sum() / valid.sum().clamp(min=1)
            be_loss = (be_loss * valid).sum() / valid.sum().clamp(min=1)
            boundary_loss = bs_loss + be_loss

            total_loss = crf_loss + self.boundary_loss_weight * boundary_loss
            return {"loss": total_loss, "crf_loss": crf_loss.detach(),
                    "boundary_loss": boundary_loss.detach(), "logits": emissions}
        else:
            decoded = self.crf.decode(emissions, mask=mask)
            return {"logits": emissions, "decoded": decoded,
                    "boundary_start_prob": torch.sigmoid(start_logits),
                    "boundary_end_prob": torch.sigmoid(end_logits)}


## 4. Shared evaluation helpers

`BoundaryAttentionTrainer` and `compute_metrics_fn` are unchanged from your notebook (needed for the
CRF model, which decodes through `model.crf.decode` rather than a plain argmax).

`compute_metrics_fn_entity_span` is **new**: it drives early-stopping / checkpoint selection while
training `LukeForEntitySpanClassification`, using a span-level accuracy/F1 that matches what that model
actually optimizes. It is a training-time convenience metric only — it never touches the final reported
comparison numbers.

`bio_tags_from_spans` is also **new**: it converts a list of non-overlapping `(char_start, char_end,
type)` spans into word-level BIO tags, used to feed both the gold entities and LUKE's final decoded
predictions into the unchanged seqeval/nervaluate scoring calls below.

`spans_overlap`, `classify_boundary`, and `run_error_analysis` are your exact matching/taxonomy logic,
unchanged. `_decode_tag_ids_to_spans` is also unchanged (still used by the RoBERTa-CRF path).
`predict_spans_with_offsets_crf` uses the RoBERTa fast tokenizer's native offsets, unchanged. LUKE gets
its own span-decoding function, `decode_entity_spans`, in section 9 — its native output is already
spans, so it doesn't need `_decode_tag_ids_to_spans` at all.


In [24]:
class BoundaryAttentionTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(
            input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],
            labels=inputs["labels"], boundary_start=inputs["boundary_start"], boundary_end=inputs["boundary_end"],
        )
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        model.eval()
        with torch.no_grad():
            loss_out = model(
                input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],
                labels=inputs["labels"], boundary_start=inputs["boundary_start"], boundary_end=inputs["boundary_end"],
            )
            loss = loss_out["loss"].detach()
            decode_out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
            decoded = decode_out["decoded"]
            max_len = inputs["labels"].shape[1]
            padded = torch.full((len(decoded), max_len), -100, dtype=torch.long)
            for i, seq in enumerate(decoded):
                padded[i, :len(seq)] = torch.tensor(seq, dtype=torch.long)
        if prediction_loss_only:
            return (loss, None, None)
        return (loss, padded, inputs["labels"])


def compute_metrics_fn(id2label):
    """Unchanged -- expects preds already as decoded tag ids (CRF model's custom trainer)."""
    def compute_metrics(eval_pred):
        preds, labels = eval_pred
        true_list, pred_list = [], []
        for pred_seq, label_seq in zip(preds, labels):
            true_seq, pred_seq_ = [], []
            for p_id, l_id in zip(pred_seq, label_seq):
                if l_id == -100:
                    continue
                true_seq.append(id2label[int(l_id)])
                pred_seq_.append(id2label[int(p_id)] if int(p_id) != -100 else "O")
            true_list.append(true_seq)
            pred_list.append(pred_seq_)
        return {
            "precision": precision_score(true_list, pred_list),
            "recall": recall_score(true_list, pred_list),
            "f1": f1_score(true_list, pred_list),
        }
    return compute_metrics


def spans_overlap(a_start, a_end, b_start, b_end):
    return a_start < b_end and b_start < a_end


def classify_boundary(g_start, g_end, p_start, p_end):
    if p_start <= g_start and p_end >= g_end:
        return "over_span"
    if p_start >= g_start and p_end <= g_end:
        return "under_span"
    return "shifted"


def predict_spans_with_offsets_crf(text, tokenizer, model, max_length=MAX_LENGTH):
    """Unchanged from your notebook -- CRF-decode path, native fast-tokenizer offsets."""
    enc = tokenizer(text, max_length=max_length, truncation=True,
                     return_offsets_mapping=True, return_tensors="pt")
    offsets = enc.pop("offset_mapping")[0].tolist()
    enc = {k: v.to(device) for k, v in enc.items()}
    model.eval()
    with torch.no_grad():
        out = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
    decoded = out["decoded"][0]
    return _decode_tag_ids_to_spans(offsets, decoded)


def _decode_tag_ids_to_spans(offsets, decoded, id2label_override=None):
    """Shared span-building logic used by every predict_spans_with_offsets_* wrapper --
    identical to the body inside your original predict_spans_with_offsets."""
    _id2label = id2label_override if id2label_override is not None else id2label
    spans = []
    cur_start, cur_end, cur_type = None, None, None
    for (char_start, char_end), tag_id in zip(offsets, decoded):
        if char_start == char_end:
            continue
        tag = _id2label[tag_id]
        if tag == "O":
            if cur_type is not None:
                spans.append((cur_start, cur_end, cur_type))
                cur_type = None
            continue
        prefix, etype = tag.split("-", 1)
        if prefix == "B" or etype != cur_type:
            if cur_type is not None:
                spans.append((cur_start, cur_end, cur_type))
            cur_start, cur_end, cur_type = char_start, char_end, etype
        else:
            cur_end = char_end
    if cur_type is not None:
        spans.append((cur_start, cur_end, cur_type))
    return spans


def run_error_analysis(test_tasks, predict_fn, sentence_idx_offset=0):
    """Same matching/taxonomy logic as your notebook's run_error_analysis, unchanged -- takes
    predict_fn(text) -> spans so it can drive any model."""
    confusion = defaultdict(lambda: defaultdict(int))
    error_rows = []

    for local_idx, task in enumerate(test_tasks):
        idx = sentence_idx_offset + local_idx
        text = task["text"]
        gold_spans = [(s, e, t) for s, e, t in task["entities"]]
        pred_spans = predict_fn(text)

        matched_pred, unmatched_gold = set(), []
        for g_start, g_end, g_type in gold_spans:
            match = None
            for p_idx, (p_start, p_end, p_type) in enumerate(pred_spans):
                if p_idx in matched_pred:
                    continue
                if p_start == g_start and p_end == g_end:
                    match = (p_idx, p_type)
                    break
            if match:
                p_idx, p_type = match
                matched_pred.add(p_idx)
                confusion[g_type][p_type] += 1
                if g_type != p_type:
                    error_rows.append({"sentence_idx": idx, "text": text[g_start:g_end], "context": text,
                                        "gold_type": g_type, "pred_type": p_type,
                                        "error_kind": "type_confusion", "boundary_kind": "exact"})
            else:
                unmatched_gold.append((g_start, g_end, g_type))

        unmatched_pred = [(p_idx, p_start, p_end, p_type)
                           for p_idx, (p_start, p_end, p_type) in enumerate(pred_spans)
                           if p_idx not in matched_pred]

        overlap_pairs = []
        for gi, (g_start, g_end, g_type) in enumerate(unmatched_gold):
            for p_idx, p_start, p_end, p_type in unmatched_pred:
                if spans_overlap(g_start, g_end, p_start, p_end):
                    ov_len = min(g_end, p_end) - max(g_start, p_start)
                    overlap_pairs.append((ov_len, gi, p_idx, g_start, g_end, g_type, p_start, p_end, p_type))
        overlap_pairs.sort(key=lambda x: -x[0])

        gold_boundary_matched, pred_boundary_matched = set(), set()
        for ov_len, gi, p_idx, g_start, g_end, g_type, p_start, p_end, p_type in overlap_pairs:
            if gi in gold_boundary_matched or p_idx in pred_boundary_matched:
                continue
            gold_boundary_matched.add(gi)
            pred_boundary_matched.add(p_idx)
            confusion[g_type]["MISSED"] += 1
            confusion["SPURIOUS"][p_type] += 1
            kind = classify_boundary(g_start, g_end, p_start, p_end)
            error_rows.append({"sentence_idx": idx, "text": text[g_start:g_end], "context": text,
                                "gold_type": g_type, "pred_type": p_type,
                                "error_kind": "boundary_error", "boundary_kind": kind})

        for gi, (g_start, g_end, g_type) in enumerate(unmatched_gold):
            if gi in gold_boundary_matched:
                continue
            is_fragment = any(spans_overlap(g_start, g_end, p_start, p_end) for _, p_start, p_end, _ in unmatched_pred)
            confusion[g_type]["MISSED"] += 1
            error_rows.append({"sentence_idx": idx, "text": text[g_start:g_end], "context": text,
                                "gold_type": g_type, "pred_type": "MISSED",
                                "error_kind": "true_miss" if not is_fragment else "boundary_error",
                                "boundary_kind": "fragmented" if is_fragment else ""})

        for p_idx, p_start, p_end, p_type in unmatched_pred:
            if p_idx in pred_boundary_matched:
                continue
            is_fragment = any(spans_overlap(p_start, p_end, g_start, g_end) for g_start, g_end, _ in unmatched_gold)
            confusion["SPURIOUS"][p_type] += 1
            error_rows.append({"sentence_idx": idx, "text": text[p_start:p_end], "context": text,
                                "gold_type": "SPURIOUS", "pred_type": p_type,
                                "error_kind": "true_spurious" if not is_fragment else "boundary_error",
                                "boundary_kind": "fragmented" if is_fragment else ""})

    gold_total_direct = sum(len(t["entities"]) for t in test_tasks)
    n_correct = sum(confusion[t].get(t, 0) for t in confusion if t != "SPURIOUS")
    n_type_confused = sum(1 for r in error_rows if r["error_kind"] == "type_confusion")
    n_boundary = sum(1 for r in error_rows if r["error_kind"] == "boundary_error" and r["gold_type"] != "SPURIOUS")
    n_true_miss = sum(1 for r in error_rows if r["error_kind"] == "true_miss")
    accounted = n_correct + n_type_confused + n_boundary + n_true_miss
    assert gold_total_direct == accounted, (
        f"MISMATCH in error analysis -- gold={gold_total_direct}, accounted={accounted}."
    )
    return confusion, error_rows


def confusion_to_dataframe(confusion):
    types = sorted({t for row in confusion.values() for t in row} | set(confusion.keys()))
    data = {p: [confusion[g].get(p, 0) for g in types] for p in types}
    return pd.DataFrame(data, index=types)


def error_taxonomy_summary(test_tasks, error_rows):
    gold_total = sum(len(t["entities"]) for t in test_tasks)
    n_type_confused = sum(1 for r in error_rows if r["error_kind"] == "type_confusion")
    n_boundary = sum(1 for r in error_rows if r["error_kind"] == "boundary_error" and r["gold_type"] != "SPURIOUS")
    n_true_miss = sum(1 for r in error_rows if r["error_kind"] == "true_miss")
    n_true_spurious = sum(1 for r in error_rows if r["error_kind"] == "true_spurious")
    n_correct = gold_total - n_type_confused - n_boundary - n_true_miss
    return {
        "gold_entities": gold_total,
        "correct": n_correct,
        "type_confusions": n_type_confused,
        "boundary_errors": n_boundary,
        "true_misses": n_true_miss,
        "true_spurious": n_true_spurious,
    }

def compute_metrics_fn_entity_span(eval_pred):
    """NEW. Metric used only to drive early stopping / checkpoint selection while training
    LukeForEntitySpanClassification. This is a SPAN-level accuracy/F1 over candidate spans
    (does the model assign the right entity-type-or-O to each candidate span?), which is the
    metric that actually matches this model's loss function -- it is NOT the final reported
    number. The final reported comparison against RoBERTa-CRF still goes through the seqeval /
    nervaluate BIO-tag pipeline below (see `bio_tags_from_spans` + the Fold 2 test-set evaluation
    cell), so this function has zero influence on the numbers in the comparison table."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    valid = labels != -100
    correct = ((preds == labels) & valid).sum()
    total = valid.sum()
    span_acc = float(correct) / float(total) if total > 0 else 0.0

    # Span-level F1 over the "is this span an entity at all" question (collapses all entity
    # types to a single positive class), which is a more informative early-stopping signal than
    # raw accuracy given the heavy O-class imbalance among candidate spans.
    pred_pos = (preds != 0) & valid
    gold_pos = (labels != 0) & valid
    tp = (pred_pos & gold_pos).sum()
    fp = (pred_pos & ~gold_pos).sum()
    fn = (~pred_pos & gold_pos).sum()
    precision = float(tp) / float(tp + fp) if (tp + fp) > 0 else 0.0
    recall = float(tp) / float(tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"span_accuracy": span_acc, "span_precision": precision, "span_recall": recall, "f1": f1}


def deoverlap_spans(spans):
    """Resolves overlapping spans into a flat, non-overlapping set. Needed because gold
    annotations can contain nested/overlapping entities (e.g. "Bank of America" ORG containing
    "America" GPE) -- normal for the source annotation tool, but not representable in a flat BIO
    tag sequence, which has no way to say two labels apply to overlapping text.

    Resolution policy: keep the LONGEST span; among equal-length spans, keep the one with the
    EARLIEST start (deterministic, independent of input order). This keeps the encompassing/outer
    entity and drops the nested one -- but ONLY for the purposes of this flat BIO conversion used
    for seqeval/nervaluate scoring. It does not touch `task["entities"]` anywhere else, so
    run_error_analysis (which handles overlap via its own spans_overlap/boundary-error logic, not
    a flat tag sequence) still sees every original gold entity, nested or not."""
    spans_sorted = sorted(spans, key=lambda s: (-(s[1] - s[0]), s[0]))

    def overlaps(a, b):
        return a[0] < b[1] and b[0] < a[1]

    selected = []
    for s in spans_sorted:
        if any(overlaps(s, sel) for sel in selected):
            continue
        selected.append(s)
    return sorted(selected, key=lambda s: s[0])


def bio_tags_from_spans(words, spans):
    """Converts a list of (char_start, char_end, entity_type) spans into a per-WORD BIO tag
    sequence aligned to `words`. Spans are de-overlapped internally via `deoverlap_spans` before
    tagging -- rather than asserting on overlap -- because gold annotations can legitimately
    contain nested/overlapping entities. This is a no-op for LUKE's predicted spans (already
    non-overlapping via NMS in decode_entity_spans) and only actually resolves anything when
    called on gold entities. A word is tagged B- if its char_start matches the (post-deoverlap)
    span's own start, I- for every other word fully inside the span, else O."""
    spans = deoverlap_spans(spans)
    tags = ["O"] * len(words)
    for (s_start, s_end, s_type) in spans:
        first = True
        for i, (w_text, w_start, w_end) in enumerate(words):
            if w_start >= s_start and w_end <= s_end and w_end > w_start:
                tags[i] = f"B-{s_type}" if first else f"I-{s_type}"
                first = False
    return tags



## 5. LUKE entity-aware span classification: candidate spans + gold-label assignment

`LukeForEntitySpanClassification` is LUKE's own official mode for NER, used in the original LUKE paper
and its released fine-tuning scripts. Unlike a token-classification head, it does not tag individual
sub-word tokens with BIO labels — instead, it classifies **candidate spans** (contiguous runs of words)
directly as one of the entity types, or `O`. The entity-aware Transformer encoder inside LUKE embeds
each candidate span itself (via LUKE's entity vocabulary and entity position embeddings), which is the
architectural feature this whole comparison is trying to evaluate faithfully — this is a genuinely
different modeling approach from the plain token-classification baseline used previously, not just a
different label format.

Concretely, for each sentence:

1. **Split into words with character offsets** (`split_words_with_offsets`) — a simple whitespace/
   punctuation-tolerant splitter. LUKE's span classifier operates over *words*, not sub-word tokens, so
   this is the correct unit here (unlike `tokenize_and_align`'s sub-word BIO alignment for RoBERTa-CRF).
2. **Enumerate every candidate span of 1..`MAX_MENTION_LENGTH` contiguous words** (`enumerate_candidate_spans`).
   A 20-word sentence with `MAX_MENTION_LENGTH=16` produces on the order of a few hundred candidates.
3. **Assign a gold label to each candidate** (`assign_span_labels`): a candidate gets the entity's type
   only if its `(char_start, char_end)` matches a gold entity's span **exactly** — partial overlaps are
   labeled `O`, same as the original LUKE fine-tuning setup (span classification is a strict-boundary
   task by construction, since there's no B-/I- mechanism to represent "this token is part of a bigger
   entity" for a partially-overlapping candidate).

Because gold-label assignment is exact-match, any gold entity longer than `MAX_MENTION_LENGTH` words can
**never** appear as a candidate and therefore can never be predicted correctly — the verification cell
below checks for this explicitly, before any training happens, so this isn't a silent failure mode.


In [25]:
import re

tokenizer_luke = LukeTokenizer.from_pretrained(LUKE_MODEL_NAME, task="entity_span_classification")


# Character-class-based tokenizer instead of whitespace-based. Three alternatives, tried in order:
#   1. a run of letters, optionally with an internal apostrophe/hyphen ("O'Brien", "Indo-Pak", "don't")
#   2. a run of digits, optionally with internal '.' or ',' ("2045", "12,000", "3.14")
#   3. fallback: ANY single non-whitespace character as its own one-character token
# Branch 3 is what fixes your bug reports: a comma, a straight or curly quote ( " ' “ ” ‘ ’ « » ),
# a parenthesis, a percent sign, an em-dash -- each becomes its own token instead of fusing onto
# the adjacent word, so gold spans that stop right before punctuation (which is how Label Studio
# highlights normally work) now land exactly on a word-token boundary.
_WORD_TOKEN_RE = re.compile(
    r"[^\W\d_]+(?:['\u2019\-][^\W\d_]+)*"   # letters, allowing one internal ' / \u2019(') / -
    r"|\d+(?:[.,]\d+)*"                       # digits, allowing internal . or ,
    r"|[^\s]"                                 # fallback: any other single non-whitespace char
)


def split_words_with_offsets(text):
    """Replacement for the old `\\S+` splitter. `[^\\W\\d_]` is 'a word character that is not a
    digit and not underscore' -- i.e. Unicode letters, matched with Python's Unicode-aware `\\w`,
    so this covers accented Latin, Cyrillic, etc. without needing an explicit character-range list.
    Punctuation and symbols (including Unicode curly quotes, which are NOT `\\w`) always fall
    through to the single-char fallback branch, so they are never fused onto a neighboring word --
    this is what makes 'Germany,' split into ('Germany', s, e) + (',', e, e+1), and what makes a
    quoted title split into (quote-char) + (word) + (word) + ... + (quote-char) instead of one
    giant token with the quote marks glued on."""
    return [(m.group(), m.start(), m.end()) for m in _WORD_TOKEN_RE.finditer(text)]


def enumerate_candidate_spans(words, max_mention_length=MAX_MENTION_LENGTH):
    """Every contiguous span of 1..max_mention_length words, as (char_start, char_end) pairs --
    the `entity_spans` format LukeTokenizer expects."""
    spans = []
    for i in range(len(words)):
        for j in range(i, min(i + max_mention_length, len(words))):
            spans.append((words[i][1], words[j][2]))
    return spans


def assign_span_labels(candidate_spans, entities):
    """Exact-match gold labeling: a candidate span gets the entity's type only if its
    (char_start, char_end) matches a gold entity's span exactly; every other candidate (including
    partial overlaps) is labeled O. This is the standard label-assignment rule for span
    classification -- there is no B-/I- mechanism to represent partial containment."""
    gold_lookup = {(s, e): span_label2id[t] for (s, e, t) in entities}
    return [gold_lookup.get(sp, span_label2id["O"]) for sp in candidate_spans]


def sample_negatives(candidate_spans, labels, ratio=TRAIN_NEG_SAMPLE_RATIO, seed=SEED):
    """TRAIN/VAL ONLY (never called for the held-out test set). Keeps every positive (true-entity)
    span; randomly subsamples negative ('O') spans down to at most `ratio` per positive span (or
    all negatives, if fewer exist), to keep per-sentence candidate counts and class imbalance
    tractable for backprop. Order is not preserved (re-sorted by start offset) -- irrelevant here
    since spans are consumed as an unordered set by the tokenizer/model."""
    rng = random.Random(seed)
    pos_idx = [i for i, l in enumerate(labels) if l != span_label2id["O"]]
    neg_idx = [i for i, l in enumerate(labels) if l == span_label2id["O"]]
    n_keep_neg = min(len(neg_idx), max(len(pos_idx) * ratio, ratio))
    kept_neg = rng.sample(neg_idx, n_keep_neg) if n_keep_neg < len(neg_idx) else neg_idx
    keep_idx = sorted(pos_idx + kept_neg)
    return [candidate_spans[i] for i in keep_idx], [labels[i] for i in keep_idx]


def build_entity_span_encoding(task, tokenizer, apply_neg_sampling, max_entity_length=MAX_ENTITY_LENGTH, seed=SEED):
    """Builds one LukeForEntitySpanClassification training/val example. `apply_neg_sampling=True`
    for TRAIN and VAL (early-stopping proxy metric only); the held-out TEST set is scored through a
    separate, non-truncating, non-sampling inference path (`decode_entity_spans`, section 9) and
    never goes through this function -- so nothing here can inflate the final reported numbers."""
    text = task["text"]
    words = split_words_with_offsets(text)
    candidate_spans = enumerate_candidate_spans(words)
    labels = assign_span_labels(candidate_spans, task["entities"])

    if apply_neg_sampling:
        candidate_spans, labels = sample_negatives(candidate_spans, labels, seed=seed)

    if len(candidate_spans) > max_entity_length:
        # Keep every positive span first; only negatives get dropped by this fallback truncation.
        pos = [(sp, l) for sp, l in zip(candidate_spans, labels) if l != span_label2id["O"]]
        neg = [(sp, l) for sp, l in zip(candidate_spans, labels) if l == span_label2id["O"]]
        keep = (pos + neg)[:max_entity_length]
        candidate_spans, labels = [k[0] for k in keep], [k[1] for k in keep]

    enc = tokenizer(text, entity_spans=candidate_spans, max_length=MAX_LENGTH,
                     padding="max_length", max_entity_length=max_entity_length,
                     truncation=True, return_tensors=None)
    enc["labels"] = labels + [-100] * (max_entity_length - len(labels))
    return enc


class NERDatasetLukeSpan(Dataset):
    """input_ids / attention_mask / entity_ids / entity_attention_mask / entity_position_ids /
    entity_start_positions / entity_end_positions / labels -- whatever fields LukeTokenizer
    produced for entity_span_classification. Used for TRAIN and VAL only."""
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        e = self.encodings[idx]
        return {k: torch.tensor(v, dtype=torch.long) for k, v in e.items()}


print("LUKE tokenizer loaded (entity_span_classification mode):", LUKE_MODEL_NAME)


LUKE tokenizer loaded (entity_span_classification mode): studio-ousia/luke-base


## 6. Verify span-candidate coverage *before* training

Required check, per your instructions: confirm the candidate-span construction can actually represent
the gold entities before spending any compute on training. Two checks, run on the reproduced Fold 2
`test_tasks` (the set the final numbers depend on) as well as `train_tasks`/`val_tasks`:

1. **Length coverage**: every gold entity's span must be `<= MAX_MENTION_LENGTH` words, or it can
   *never* be produced as a candidate and is therefore unreachable by construction, regardless of how
   well the model trains. Any such entities are reported explicitly, by tag, rather than silently
   scored as misses later.
2. **Exact-match reachability**: for every gold entity, its exact `(char_start, char_end)` must appear
   in that sentence's candidate-span list (`enumerate_candidate_spans`) — a lower-level check that
   `split_words_with_offsets`'s word boundaries actually land on the gold entity's own start/end
   offsets (entities that start or end mid-word relative to the whitespace splitter would otherwise
   silently fail to be labeled correctly).


In [26]:
def check_span_coverage(tasks_subset, name, max_mention_length=MAX_MENTION_LENGTH):
    n_entities = 0
    too_long = defaultdict(int)
    unreachable = []
    for task in tasks_subset:
        text = task["text"]
        words = split_words_with_offsets(text)
        candidate_set = set(enumerate_candidate_spans(words, max_mention_length))
        for (ent_start, ent_end, ent_label) in task["entities"]:
            n_entities += 1
            ent_word_len = sum(1 for (_, w_s, w_e) in words if w_s >= ent_start and w_e <= ent_end and w_e > w_s)
            if ent_word_len > max_mention_length:
                too_long[ent_label] += 1
            elif (ent_start, ent_end) not in candidate_set:
                unreachable.append((text, ent_start, ent_end, ent_label))

    print(f"=== Span-coverage check: {name} ({len(tasks_subset)} sentences, {n_entities} gold entities) ===")
    if too_long:
        total_too_long = sum(too_long.values())
        print(f"  (!) {total_too_long} gold entities exceed MAX_MENTION_LENGTH={max_mention_length} words "
              f"and can NEVER be predicted correctly by construction. By tag: {dict(too_long)}")
    else:
        print(f"  All gold entities are <= MAX_MENTION_LENGTH={max_mention_length} words.")
    if unreachable:
        print(f"  (!) {len(unreachable)} gold entities are within the length limit but their exact "
              f"(char_start, char_end) was not produced by the word splitter -- inspect these:")
        for text, s, e, label in unreachable[:5]:
            print(f"      [{label:6s}] {text[s:e]!r}  in: {text[:80]!r}...")
    else:
        print("  All within-length gold entities are exactly reachable as candidate spans.")
    print()
    return too_long, unreachable


_ = check_span_coverage(test_tasks, "Fold 2 test set")
_ = check_span_coverage(train_tasks, "Fold 2 train pool")
_ = check_span_coverage(val_tasks, "Fold 2 val pool")


=== Span-coverage check: Fold 2 test set (4808 sentences, 13931 gold entities) ===
  (!) 6 gold entities exceed MAX_MENTION_LENGTH=16 words and can NEVER be predicted correctly by construction. By tag: {'MONEY': 1, 'WOA': 4, 'LAW': 1}
  (!) 30 gold entities are within the length limit but their exact (char_start, char_end) was not produced by the word splitter -- inspect these:
      [PER   ] 'Seth Londo'  in: 'That ’s all we have time for today , but there are multiple other notables invol'...
      [LOC   ] 'India'  in: "The summit concluded with optimism regarding India's artificial-intelligence fut"...
      [LOC   ] 'India'  in: 'Union Minister Piyush Goyal stated that Industrial Internet of Things innovation'...
      [LOC   ] 'India'  in: 'Union Minister Pralhad Joshi stated that green hydrogen will play a crucial role'...
      [LOC   ] 'India'  in: 'Union Minister Ashwini Vaishnaw stated that cybersecurity skills remain essentia'...

=== Span-coverage check: Fold 2 train pool 

In [20]:
check_span_coverage(test_tasks, "Fold 2 test")
check_span_coverage(train_tasks, "Fold 2 train")
check_span_coverage(val_tasks, "Fold 2 val")

=== Span-coverage check: Fold 2 test (4808 sentences, 13931 gold entities) ===
  (!) 6 gold entities exceed MAX_MENTION_LENGTH=16 words and can NEVER be predicted correctly by construction. By tag: {'MONEY': 1, 'WOA': 4, 'LAW': 1}
  (!) 30 gold entities are within the length limit but their exact (char_start, char_end) was not produced by the word splitter -- inspect these:
      [PER   ] 'Seth Londo'  in: 'That ’s all we have time for today , but there are multiple other notables invol'...
      [LOC   ] 'India'  in: "The summit concluded with optimism regarding India's artificial-intelligence fut"...
      [LOC   ] 'India'  in: 'Union Minister Piyush Goyal stated that Industrial Internet of Things innovation'...
      [LOC   ] 'India'  in: 'Union Minister Pralhad Joshi stated that green hydrogen will play a crucial role'...
      [LOC   ] 'India'  in: 'Union Minister Ashwini Vaishnaw stated that cybersecurity skills remain essentia'...

=== Span-coverage check: Fold 2 train (17313 se

(defaultdict(int, {}),
 [("The summit concluded with optimism regarding India's pharmaceutical future.",
   45,
   50,
   'LOC'),
  ('Chris Sosa on Substack argues that last night ’s results show a GOP that is eliminating dissent and a Democratic party that is moving towards freedom from foreign control : As the GOP continues to consolidate into something between a personal party of the sitting president and a purchasable entity by major super PACs , a populist revolt continues in the Democratic Party .',
   35,
   44,
   'TIME'),
  ("Union Minister Pralhad Joshi stated that clean-energy expansion remains a strategic priority for India's long-term development goals.",
   97,
   102,
   'LOC'),
  ("The summit concluded with optimism regarding India's energy-storage future.",
   45,
   50,
   'LOC'),
  ("The summit concluded with optimism regarding India's edge-computing technology future.",
   45,
   50,
   'LOC'),
  ("The summit concluded with optimism regarding India's smart-grid tech

## 7. Build the LUKE entity-span train / val datasets on the reproduced Fold 2 split

Train and val examples are built with `apply_neg_sampling=True` (see section 5) to keep candidate-span
counts tractable for backprop. The held-out Fold 2 **test set is deliberately not built here** — it is
scored later (section 9) through a separate inference path that evaluates *every* candidate span with
no sampling and no truncation, so the model-selection convenience above cannot influence the final
reported test numbers.


In [27]:
set_seed(SEED)

train_encodings_luke = [build_entity_span_encoding(t, tokenizer_luke, apply_neg_sampling=True, seed=SEED)
                         for t in train_tasks]
val_encodings_luke   = [build_entity_span_encoding(t, tokenizer_luke, apply_neg_sampling=True, seed=SEED)
                         for t in val_tasks]

train_ds_luke = NERDatasetLukeSpan(train_encodings_luke)
val_ds_luke   = NERDatasetLukeSpan(val_encodings_luke)

n_pos_train = sum(1 for e in train_encodings_luke for l in e["labels"] if l not in (-100, span_label2id["O"]))
n_pos_val   = sum(1 for e in val_encodings_luke for l in e["labels"] if l not in (-100, span_label2id["O"]))
print(f"LUKE entity-span datasets built -- train={len(train_ds_luke)} sentences "
      f"({n_pos_train} positive spans after neg-sampling) | "
      f"val={len(val_ds_luke)} sentences ({n_pos_val} positive spans after neg-sampling)")


LUKE entity-span datasets built -- train=17313 sentences (50536 positive spans after neg-sampling) | val=1923 sentences (5582 positive spans after neg-sampling)


## 8. Train LUKE-base — `LukeForEntitySpanClassification`, official span-classification head

`LukeForEntitySpanClassification.from_pretrained(...)` only — no `BoundaryGuidedAttention`, no CRF
layer, no boundary loss, no custom `Trainer` subclass. Training hyperparameters (`LEARNING_RATE`,
`BATCH_SIZE`, `GRAD_ACCUM_STEPS`, `NUM_EPOCHS`, `WARMUP_RATIO`, `WEIGHT_DECAY`, early stopping patience)
are identical to your k-fold loop's `TrainingArguments`. Trains **only** on the reproduced Fold 2
`train_tasks`/`val_tasks` — not on the other 4 folds, and not on the test set. `metric_for_best_model`
now points at the span-level `f1` returned by `compute_metrics_fn_entity_span` (section 4), the metric
that actually matches this model's classification head — not the seqeval BIO metric used to select the
previous token-classification LUKE run, since that metric doesn't apply to a span classifier's raw
per-batch output.


In [28]:
set_seed(SEED)

model_luke = LukeForEntitySpanClassification.from_pretrained(
    LUKE_MODEL_NAME,
    num_labels=len(span_label2id),
    id2label=span_id2label,
    label2id=span_label2id,
).to(device)

luke_output_dir = os.path.join(OUTPUT_DIR, "luke_fold2_span")
args_luke = TrainingArguments(
    output_dir=luke_output_dir,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to=[],
    save_total_limit=1,
)

trainer_luke = Trainer(
    model=model_luke,
    args=args_luke,
    train_dataset=train_ds_luke,
    eval_dataset=val_ds_luke,
    compute_metrics=compute_metrics_fn_entity_span,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PAT)],
)

trainer_luke.train()


Some weights of LukeForEntitySpanClassification were not initialized from the model checkpoint at studio-ousia/luke-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Span Accuracy,Span Precision,Span Recall,F1
1,0.082200,0.070957,0.980238,0.970739,0.980652,0.975671
2,0.059900,0.056300,0.983796,0.980150,0.981906,0.981027
3,0.053500,0.053499,0.984465,0.978822,0.985310,0.982055


TrainOutput(global_step=1626, training_loss=0.17281583424423894, metrics={'train_runtime': 1949.8149, 'train_samples_per_second': 26.638, 'train_steps_per_second': 0.834, 'total_flos': 8548048859415552.0, 'train_loss': 0.17281583424423894, 'epoch': 3.0})

## 9. Evaluate LUKE-base (entity-span classification) on the Fold 2 test set

Every candidate span in every test sentence is scored — no negative sampling, no truncation (chunked
into `MAX_ENTITY_LENGTH`-sized forward passes instead, so sentences with more candidates than that just
take multiple passes rather than losing candidates). Overlapping non-`O` predictions are resolved by
greedy non-maximum suppression (`decode_entity_spans`) into a final non-overlapping span set, which is
then converted to word-level BIO tags (`bio_tags_from_spans`) — for both the gold entities and LUKE's
predictions — purely so the existing seqeval/nervaluate calls can score it on the same footing as the
RoBERTa-CRF model below. The error-taxonomy pipeline (section 4/next cell) uses the raw decoded spans
directly, without going through BIO at all.


In [29]:
def predict_entity_spans_luke(text, tokenizer, model, id2label,
                               max_mention_length=MAX_MENTION_LENGTH, chunk_size=MAX_ENTITY_LENGTH):
    """Full, non-truncating, non-sampled inference over every candidate span in `text`. Chunks
    candidates into batches of `chunk_size` (LukeTokenizer's entity side has a fixed max length per
    forward pass), runs one forward pass per chunk, and returns (span, predicted_label_id,
    confidence) for every single candidate -- nothing is dropped or sampled here, unlike training."""
    words = split_words_with_offsets(text)
    candidate_spans = enumerate_candidate_spans(words, max_mention_length)
    if not candidate_spans:
        return []

    model.eval()
    results = []
    for i in range(0, len(candidate_spans), chunk_size):
        chunk = candidate_spans[i:i + chunk_size]
        enc = tokenizer(text, entity_spans=chunk, max_length=MAX_LENGTH,
                         max_entity_length=len(chunk), truncation=True, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
        probs = torch.softmax(out.logits[0], dim=-1)
        conf, pred_id = probs.max(dim=-1)
        for sp, p, c in zip(chunk, pred_id.tolist(), conf.tolist()):
            results.append((sp, p, c))
    return results


def decode_entity_spans(text, tokenizer, model, id2label,
                         max_mention_length=MAX_MENTION_LENGTH, chunk_size=MAX_ENTITY_LENGTH):
    """Greedy non-maximum suppression over every candidate span's prediction: rank all non-O
    candidates by confidence, keep the highest, discard every remaining candidate that overlaps it
    in character space, repeat. Produces the final non-overlapping (char_start, char_end,
    entity_type) span set that LUKE's span-classification head is actually meant to output."""
    raw = predict_entity_spans_luke(text, tokenizer, model, id2label, max_mention_length, chunk_size)
    non_o = [((s, e), pid, conf) for (s, e), pid, conf in raw if id2label[pid] != "O"]
    non_o.sort(key=lambda x: -x[2])

    def overlaps(a, b):
        return a[0] < b[1] and b[0] < a[1]

    selected = []
    for (s, e), pid, conf in non_o:
        if any(overlaps((s, e), (ss, ee)) for ss, ee, _ in selected):
            continue
        selected.append((s, e, id2label[pid]))
    selected.sort(key=lambda x: x[0])
    return selected


true_list_luke, pred_list_luke = [], []
for n_done, task in enumerate(test_tasks, start=1):
    text = task["text"]
    words = split_words_with_offsets(text)
    gold_spans = [(s, e, t) for s, e, t in task["entities"]]
    pred_spans = decode_entity_spans(text, tokenizer_luke, model_luke, span_id2label)

    true_list_luke.append(bio_tags_from_spans(words, gold_spans))
    pred_list_luke.append(bio_tags_from_spans(words, pred_spans))

    if n_done % 200 == 0 or n_done == len(test_tasks):
        print(f"  [LUKE inference] {n_done}/{len(test_tasks)} Fold {FOLD_NUMBER} test sentences decoded")

gold_total_raw = sum(len(t["entities"]) for t in test_tasks)
gold_total_deoverlapped = sum(len(deoverlap_spans([(s, e, t) for s, e, t in task["entities"]]))
                               for task in test_tasks)
print(f"[Gold overlap check] {gold_total_raw} raw gold entities -> "
      f"{gold_total_deoverlapped} after de-overlapping for BIO scoring "
      f"({gold_total_raw - gold_total_deoverlapped} nested entities dropped from seqeval/nervaluate "
      f"numbers only -- run_error_analysis below still scores all {gold_total_raw}).")

luke_f1        = f1_score(true_list_luke, pred_list_luke)
luke_precision = precision_score(true_list_luke, pred_list_luke)
luke_recall    = recall_score(true_list_luke, pred_list_luke)
luke_accuracy  = accuracy_score(true_list_luke, pred_list_luke)

print(f"\n[LUKE-base (entity-span classification) | Fold {FOLD_NUMBER} test] "
      f"P={luke_precision:.4f} R={luke_recall:.4f} F1={luke_f1:.4f} token-acc={luke_accuracy:.4f}")
print()
print(classification_report(true_list_luke, pred_list_luke, digits=4))

from nervaluate import Evaluator
evaluator_luke = Evaluator(true_list_luke, pred_list_luke, tags=ENTITY_TAGS, loader="list")
sem_results_luke = evaluator_luke.evaluate()
if isinstance(sem_results_luke, tuple):
    sem_results_luke = sem_results_luke[0]
sem_overall_luke = sem_results_luke["overall"] if "overall" in sem_results_luke else sem_results_luke
strict_luke = sem_overall_luke["strict"]
luke_strict_f1 = strict_luke["f1"] if isinstance(strict_luke, dict) else strict_luke.f1
print(f"\nSemEval strict F1 (LUKE-base, entity-span classification): {luke_strict_f1:.4f}")


  [LUKE inference] 200/4808 Fold 2 test sentences decoded
  [LUKE inference] 400/4808 Fold 2 test sentences decoded
  [LUKE inference] 600/4808 Fold 2 test sentences decoded
  [LUKE inference] 800/4808 Fold 2 test sentences decoded
  [LUKE inference] 1000/4808 Fold 2 test sentences decoded
  [LUKE inference] 1200/4808 Fold 2 test sentences decoded
  [LUKE inference] 1400/4808 Fold 2 test sentences decoded
  [LUKE inference] 1600/4808 Fold 2 test sentences decoded
  [LUKE inference] 1800/4808 Fold 2 test sentences decoded
  [LUKE inference] 2000/4808 Fold 2 test sentences decoded
  [LUKE inference] 2200/4808 Fold 2 test sentences decoded
  [LUKE inference] 2400/4808 Fold 2 test sentences decoded
  [LUKE inference] 2600/4808 Fold 2 test sentences decoded
  [LUKE inference] 2800/4808 Fold 2 test sentences decoded
  [LUKE inference] 3000/4808 Fold 2 test sentences decoded
  [LUKE inference] 3200/4808 Fold 2 test sentences decoded
  [LUKE inference] 3400/4808 Fold 2 test sentences decoded
 

In [30]:
# --- confusion matrix + error-analysis taxonomy for LUKE, same taxonomy as your notebook ---
# predict_fn_luke returns raw decoded spans directly (LUKE's native output) -- run_error_analysis
# never needed BIO tags in the first place, only predict_spans_with_offsets_crf's BIO-decode did.
predict_fn_luke = lambda text: decode_entity_spans(text, tokenizer_luke, model_luke, span_id2label)
confusion_luke, error_rows_luke = run_error_analysis(test_tasks, predict_fn_luke)

confusion_df_luke = confusion_to_dataframe(confusion_luke)
summary_luke = error_taxonomy_summary(test_tasks, error_rows_luke)

print("=== LUKE-base (entity-span classification) error taxonomy (Fold 2 test set) ===")
for k, v in summary_luke.items():
    print(f"  {k:16s}: {v}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
confusion_df_luke.to_csv(os.path.join(OUTPUT_DIR, "confusion_matrix_luke_fold2.csv"))
pd.DataFrame(error_rows_luke).to_csv(os.path.join(OUTPUT_DIR, "error_log_luke_fold2.csv"), index=False)
print(f"\nWrote confusion_matrix_luke_fold2.csv and error_log_luke_fold2.csv to {OUTPUT_DIR}")
confusion_df_luke


=== LUKE-base (entity-span classification) error taxonomy (Fold 2 test set) ===
  gold_entities   : 13931
  correct         : 10271
  type_confusions : 549
  boundary_errors : 2856
  true_misses     : 255
  true_spurious   : 2125

Wrote confusion_matrix_luke_fold2.csv and error_log_luke_fold2.csv to ./ner_fold2_luke_comparison_output


,CARD,DATE,EVT,FAC,LANG,LAW,LOC,MISSED,MONEY,NORP,ORD,ORG,PCT,PER,PROD,QTY,SPURIOUS,TIME,WOA
CARD,727,6,0,0,0,0,0,192,1,0,1,0,0,0,1,0,0,1,0
DATE,11,1294,0,0,0,0,1,486,0,0,3,0,0,0,0,0,0,0,0
EVT,0,2,51,0,0,0,4,124,0,0,0,3,0,6,2,0,0,0,2
FAC,0,0,0,25,0,0,12,58,0,0,0,7,0,2,2,0,0,0,0
LANG,0,0,0,0,26,0,1,0,0,14,0,0,0,0,0,0,0,0,0
LAW,0,0,0,0,0,43,2,77,0,0,1,13,0,6,0,0,0,0,0
LOC,0,0,0,2,0,0,2231,255,0,7,0,20,0,13,0,0,0,0,0
MISSED,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
MONEY,1,0,0,0,0,0,0,196,234,0,0,1,1,0,0,0,0,0,0
NORP,0,0,0,0,0,0,7,52,0,966,0,8,0,7,1,0,0,0,0


## 10. Load your RoBERTa + CRF + Boundary Attention Fold 2 checkpoint (no retraining)

Loads `Souvikbasur/roberta-crf-news-ner-kfold-best` and evaluates it, as-is, on the **same** Fold 2
`test_tasks` reproduced above. The tokenizer is instantiated the same way your training loop
instantiated it (`AutoTokenizer.from_pretrained(BASE_MODEL, add_prefix_space=True)`) rather than pulled
from the repo, since the tokenizer files saved to the repo were saved *without* `add_prefix_space=True` --
using the repo copy would silently diverge from the tokenizer actually used to produce this checkpoint's
labels during training.


In [31]:
# Only prompts for a token if the repo turns out to need auth -- most public HF repos don't.
# Put your token in Colab's Secrets tab (key icon, left sidebar) as HF_TOKEN to skip the prompt entirely.
hf_token = os.environ.get("HF_TOKEN")
if hf_token is None:
    try:
        hf_hub_download(repo_id=CRF_MODEL_REPO, filename="label_map.json")
        print("Repo is public -- no token needed.")
    except Exception:
        hf_token = getpass.getpass("Enter your Hugging Face token (repo needs auth): ")

from safetensors.torch import load_file

ckpt_path  = hf_hub_download(repo_id=CRF_MODEL_REPO, filename="model.safetensors", token=hf_token)
label_path = hf_hub_download(repo_id=CRF_MODEL_REPO, filename="label_map.json", token=hf_token)

with open(label_path) as f:
    maps = json.load(f)
repo_label2id = maps["label2id"]
repo_id2label = {int(k): v for k, v in maps["id2label"].items()}

assert repo_label2id == label2id, (
    "Label map mismatch between the checkpoint repo and this notebook's ENTITY_TAGS-derived "
    "label2id -- the comparison below would not be apples-to-apples. Check ENTITY_TAGS matches "
    "what the checkpoint was trained with."
)
print("Label map from the checkpoint repo matches this notebook's label2id -- safe to proceed.")

tokenizer_roberta = AutoTokenizer.from_pretrained(BASE_MODEL, add_prefix_space=True)

crf_model = RobertaCRFBoundaryAttentionForNER(BASE_MODEL, num_labels=len(label2id)).to(device)
state_dict = load_file(ckpt_path)
missing, unexpected = crf_model.load_state_dict(state_dict, strict=False)
print("missing keys:", missing)
print("unexpected keys:", unexpected)   # both should be empty for a clean load
crf_model.eval()
print("RoBERTa-CRF-BoundaryAttention checkpoint loaded, no training performed.")


label_map.json: 0.00B [00:00, ?B/s]

Repo is public -- no token needed.


model.safetensors:   0%|          | 0.00/508M [00:00<?, ?B/s]

Label map from the checkpoint repo matches this notebook's label2id -- safe to proceed.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


missing keys: []
unexpected keys: []
RoBERTa-CRF-BoundaryAttention checkpoint loaded, no training performed.


## 11. Evaluate the RoBERTa-CRF-BoundaryAttention checkpoint — same pipeline, same test set

Uses the unchanged `BoundaryAttentionTrainer` in **prediction-only mode** (`trainer.train()` is never
called) so the seqeval numbers go through the identical decode path (`model.crf.decode`) your k-fold
loop used for every fold's held-out window.


In [32]:
test_ds_roberta = NERDataset(tokenize_and_align(test_tasks, tokenizer_roberta, label2id))

eval_only_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "crf_fold2_eval_only"),
    per_device_eval_batch_size=BATCH_SIZE,
    report_to=[],
)

eval_trainer = BoundaryAttentionTrainer(model=crf_model, args=eval_only_args)
preds_out_crf = eval_trainer.predict(test_ds_roberta)
preds_crf, labels_crf = preds_out_crf.predictions, preds_out_crf.label_ids

true_list_crf, pred_list_crf = [], []
for pred_seq, label_seq in zip(preds_crf, labels_crf):
    true_seq, pred_seq_ = [], []
    for p_id, l_id in zip(pred_seq, label_seq):
        if l_id == -100:
            continue
        true_seq.append(id2label[int(l_id)])
        pred_seq_.append(id2label[int(p_id)] if int(p_id) != -100 else "O")
    true_list_crf.append(true_seq)
    pred_list_crf.append(pred_seq_)

crf_f1        = f1_score(true_list_crf, pred_list_crf)
crf_precision = precision_score(true_list_crf, pred_list_crf)
crf_recall    = recall_score(true_list_crf, pred_list_crf)
crf_accuracy  = accuracy_score(true_list_crf, pred_list_crf)

print(f"[RoBERTa-CRF-BoundaryAttention | Fold {FOLD_NUMBER} test] "
      f"P={crf_precision:.4f} R={crf_recall:.4f} F1={crf_f1:.4f} token-acc={crf_accuracy:.4f}")
print()
print(classification_report(true_list_crf, pred_list_crf, digits=4))

evaluator_crf = Evaluator(true_list_crf, pred_list_crf, tags=ENTITY_TAGS, loader="list")
sem_results_crf = evaluator_crf.evaluate()
if isinstance(sem_results_crf, tuple):
    sem_results_crf = sem_results_crf[0]
sem_overall_crf = sem_results_crf["overall"] if "overall" in sem_results_crf else sem_results_crf
strict_crf = sem_overall_crf["strict"]
crf_strict_f1 = strict_crf["f1"] if isinstance(strict_crf, dict) else strict_crf.f1
print(f"\nSemEval strict F1 (RoBERTa-CRF-BoundaryAttention): {crf_strict_f1:.4f}")


[RoBERTa-CRF-BoundaryAttention | Fold 2 test] P=0.9444 R=0.9616 F1=0.9529 token-acc=0.9900

              precision    recall  f1-score   support

        CARD     0.9288    0.9569    0.9426       927
        DATE     0.9437    0.9694    0.9564      1765
         EVT     0.9021    0.9115    0.9067       192
         FAC     0.8000    0.9057    0.8496       106
        LANG     0.9500    0.9268    0.9383        41
         LAW     0.8627    0.9296    0.8949       142
         LOC     0.9732    0.9794    0.9763      2520
       MONEY     0.9613    0.9746    0.9679       433
        NORP     0.9641    0.9817    0.9728      1039
         ORD     0.9008    0.9793    0.9384       241
         ORG     0.9380    0.9468    0.9424      2859
         PCT     0.9430    0.9622    0.9525       344
         PER     0.9628    0.9790    0.9708      2329
        PROD     0.9044    0.9342    0.9190       243
         QTY     0.9336    0.9073    0.9202       248
        TIME     0.8541    0.9133    0.8827

In [34]:
# --- confusion matrix + error-analysis taxonomy for the CRF model, identical taxonomy code ---
predict_fn_crf = lambda text: predict_spans_with_offsets_crf(text, tokenizer_roberta, crf_model)
confusion_crf, error_rows_crf = run_error_analysis(test_tasks, predict_fn_crf)

confusion_df_crf = confusion_to_dataframe(confusion_crf)
summary_crf = error_taxonomy_summary(test_tasks, error_rows_crf)

print("=== RoBERTa-CRF-BoundaryAttention error taxonomy (Fold 2 test set) ===")
for k, v in summary_crf.items():
    print(f"  {k:16s}: {v}")

confusion_df_crf.to_csv(os.path.join(OUTPUT_DIR, "confusion_matrix_roberta_crf_fold2.csv"))
pd.DataFrame(error_rows_crf).to_csv(os.path.join(OUTPUT_DIR, "error_log_roberta_crf_fold2.csv"), index=False)
print(f"\nWrote confusion_matrix_roberta_crf_fold2.csv and error_log_roberta_crf_fold2.csv to {OUTPUT_DIR}")
confusion_df_crf


=== RoBERTa-CRF-BoundaryAttention error taxonomy (Fold 2 test set) ===
  gold_entities   : 13931
  correct         : 13339
  type_confusions : 133
  boundary_errors : 355
  true_misses     : 104
  true_spurious   : 268

Wrote confusion_matrix_roberta_crf_fold2.csv and error_log_roberta_crf_fold2.csv to ./ner_fold2_luke_comparison_output


,CARD,DATE,EVT,FAC,LANG,LAW,LOC,MISSED,MONEY,NORP,ORD,ORG,PCT,PER,PROD,QTY,SPURIOUS,TIME,WOA
CARD,886,4,0,0,0,0,0,32,0,0,1,0,0,0,1,3,0,2,0
DATE,3,1711,0,0,0,0,0,79,0,0,2,0,0,0,0,0,0,0,0
EVT,0,0,175,0,0,0,0,17,0,0,0,0,0,2,0,0,0,0,0
FAC,0,0,0,96,0,0,2,8,0,0,0,0,0,0,0,0,0,0,0
LANG,0,0,0,0,38,0,0,1,0,2,0,0,0,0,0,0,0,0,0
LAW,0,0,0,0,0,131,0,11,0,0,0,0,0,0,0,0,0,0,0
LOC,0,0,0,1,0,0,2471,48,0,2,0,3,0,2,0,0,0,0,1
MISSED,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
MONEY,0,0,0,0,0,0,0,10,422,0,0,0,0,0,1,0,0,0,0
NORP,0,0,0,0,1,0,1,16,0,1020,0,2,0,1,0,0,0,0,0


## 12. Final comparison — identical Fold 2 test set, identical evaluation pipeline


In [33]:
comparison = pd.DataFrame([
    {
        "Model": "LUKE-base (LukeForEntitySpanClassification, official span-classification head)",
        "Precision": luke_precision, "Recall": luke_recall, "F1": luke_f1,
        "SemEval strict F1": luke_strict_f1, "Token Accuracy": luke_accuracy,
        "n_test_sentences": len(test_tasks),
    },
    {
        "Model": "RoBERTa + CRF + Boundary Attention (Fold 2 checkpoint)",
        "Precision": crf_precision, "Recall": crf_recall, "F1": crf_f1,
        "SemEval strict F1": crf_strict_f1, "Token Accuracy": crf_accuracy,
        "n_test_sentences": len(test_tasks),
    },
]).set_index("Model")

comparison_path = os.path.join(OUTPUT_DIR, "fold2_model_comparison_luke.csv")
comparison.to_csv(comparison_path)

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
print(f"Both models evaluated on the same {len(test_tasks)}-sentence Fold {FOLD_NUMBER} test set.\n")
display(comparison)

winner = comparison["F1"].idxmax()
f1_gap = abs(comparison.loc[comparison.index[0], "F1"] - comparison.loc[comparison.index[1], "F1"])
print(f"\n>>> Higher seqeval F1 on the Fold {FOLD_NUMBER} test set: {winner} "
      f"(by {f1_gap:.4f} F1 points)")
print(f"Wrote {comparison_path}")


Both models evaluated on the same 4808-sentence Fold 2 test set.



,Precision,Recall,F1,SemEval strict F1,Token Accuracy,n_test_sentences
Model,,,,,,
"LUKE-base (LukeForEntitySpanClassification, official span-classification head)",0.5975,0.7379,0.6603,0.6603,0.9071,4808
RoBERTa + CRF + Boundary Attention (Fold 2 checkpoint),0.9444,0.9616,0.9529,0.9529,0.9900,4808



>>> Higher seqeval F1 on the Fold 2 test set: RoBERTa + CRF + Boundary Attention (Fold 2 checkpoint) (by 0.2926 F1 points)
Wrote ./ner_fold2_luke_comparison_output/fold2_model_comparison_luke.csv


In [35]:
# --- side-by-side error-taxonomy comparison ---
taxonomy_compare = pd.DataFrame({
    "LUKE-base": summary_luke,
    "RoBERTa-CRF-BoundaryAttention": summary_crf,
})
print("Error taxonomy, side by side (counts over the same test entities):\n")
display(taxonomy_compare)


Error taxonomy, side by side (counts over the same test entities):



,LUKE-base,RoBERTa-CRF-BoundaryAttention
gold_entities,13931,13931
correct,10271,13339
type_confusions,549,133
boundary_errors,2856,355
true_misses,255,104
true_spurious,2125,268


## Caveats worth reading before trusting the verdict above

- **This is one fold, not the k-fold mean.** A single split/fold can be an outlier. This comparison is
  only as reliable as Fold 2 is representative of the other 4 -- cross-check the
  `RoBERTa-CRF-BoundaryAttention` F1 above against the fold-2 row in your saved `kfold_summary.csv`
  (mean ± std across all 5 folds) before generalizing "which model wins" beyond this one fold.
- **`FOLD_IDX` is now 2, matching your uploaded notebook's code cell** (not the earlier markdown
  description of "fold_idx = 1"), per your explicit instruction. If `folds[2]` is *not* actually what
  produced the `Souvikbasur/roberta-crf-news-ner-kfold-best` checkpoint in your k-fold training run,
  this comparison silently stops being apples-to-apples -- re-verify against that notebook.
- **The two models are architecturally very different, on both axes now.** LUKE-base here uses its
  official *entity-aware span classification* head (`LukeForEntitySpanClassification`) -- candidate
  word-spans classified directly, with greedy NMS decoding -- while the RoBERTa model is a *sequence
  tagger* (sub-word BIO tags decoded via a CRF layer plus an auxiliary boundary-attention block), and
  was **warm-started** from an already fully-trained checkpoint rather than trained from public
  pretrained RoBERTa weights on this fold. So this table answers *"does LUKE's own official
  span-classification approach beat your full, already-trained RoBERTa+CRF+BoundaryAttention system on
  Fold 2"* -- it does not isolate whether any gap comes from the backbone, the modeling paradigm
  (span classification vs. sequence tagging), the extra architecture, or the warm start.
- **The seqeval/nervaluate numbers for LUKE are computed at WORD granularity, not sub-word.** LUKE's
  span classifier natively reasons over words; RoBERTa-CRF reasons over sub-word tokens. Both are
  converted to a common representation for scoring (LUKE via `bio_tags_from_spans` on its word-level
  candidate grid, RoBERTa-CRF via its native sub-word offsets), but they are not the *same* tokenization
  grid. The SemEval **strict F1** (entity-level exact-match, tokenization-agnostic) is the more directly
  comparable number between the two for this reason -- read the seqeval token-level numbers as a
  secondary, coarser signal.
- **`MAX_MENTION_LENGTH` (16 words) caps what LUKE can ever predict correctly.** The section-6
  verification cell reports, by tag, how many gold entities in each split exceed this length and are
  therefore structurally unreachable regardless of training quality. If any of your longer-tailed tags
  (`WOA`, `LAW`) show up there in nontrivial numbers, LUKE's recall on those tags is capped below 100%
  by construction, independent of model quality -- factor that in before concluding LUKE is
  systematically weaker on those tags.
- **`TRAIN_NEG_SAMPLE_RATIO` only affects training/model-selection, not the reported test metrics.**
  Every positive (true-entity) candidate span is always kept during training; only negative ('O')
  candidates get downsampled, and only for `train_tasks`/`val_tasks`. The Fold 2 **test set** is always
  scored on every single candidate span (chunked at inference, never sampled or truncated) -- see
  section 9 -- so this cannot inflate the F1 numbers above.
- **`ENTITY_TAGS` / label sets are separate for the two models by design.** RoBERTa-CRF uses the
  original BIO `label2id` (35 classes: `O` + `B-`/`I-` per tag); LUKE's span classifier uses
  `span_label2id` (18 classes: `O` + one class per tag, no `B-`/`I-`). This is intentional -- span
  classification has no need for the B-/I- distinction -- not a bug or an oversight.
